# **EDA**

In [ ]:
! pip install ppscore
#loading a dataset
import pandas as pd

# Load the dataset
file_path = "/content/adult_with_headers.csv"
df = pd.read_csv(file_path)

# Display basic information
print(df.info())

# Show summary statistics
print(df.describe())

# Check for missing values
print(df.isnull().sum())

# Show first few rows
df.head()

#Handle a missing values
# Handle missing values: Remove rows with missing values
df.dropna(inplace=True)

# Verify missing values are removed
print(df.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
None
                age        fnlwgt  education_num  capital_gain  capital_loss  \
count  3

In [ ]:
#Scaling numerical features
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Select numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Apply Standard Scaling
scaler = StandardScaler()
df_standard_scaled = df.copy()
df_standard_scaled[num_cols] = scaler.fit_transform(df[num_cols])

# Apply Min-Max Scaling
minmax_scaler = MinMaxScaler()
df_minmax_scaled = df.copy()
df_minmax_scaled[num_cols] = minmax_scaler.fit_transform(df[num_cols])

# Show scaled data
df_standard_scaled.head(), df_minmax_scaled.head()


(        age          workclass    fnlwgt   education  education_num  \
 0  0.030671          State-gov -1.063611   Bachelors       1.134739   
 1  0.837109   Self-emp-not-inc -1.008707   Bachelors       1.134739   
 2 -0.042642            Private  0.245079     HS-grad      -0.420060   
 3  1.057047            Private  0.425801        11th      -1.197459   
 4 -0.775768            Private  1.408176   Bachelors       1.134739   
 
         marital_status          occupation    relationship    race      sex  \
 0        Never-married        Adm-clerical   Not-in-family   White     Male   
 1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
 2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
 3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
 4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   
 
    capital_gain  capital_loss  hours_per_week  native_country  income  
 0      0.1

In [ ]:
#Encoding Cateogrical Variables
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns
cat_cols = df.select_dtypes(include=['object']).columns

# Apply One-Hot Encoding (For categories < 5 unique values)
df_encoded = pd.get_dummies(df, columns=[col for col in cat_cols if df[col].nunique() < 5], drop_first=True)

# Apply Label Encoding (For categories > 5 unique values)
label_encoders = {}
for col in [col for col in cat_cols if df[col].nunique() >= 5]:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col])
    label_encoders[col] = le  # Save the encoder for future use

# Check encoded dataset
df_encoded.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,capital_gain,capital_loss,hours_per_week,native_country,sex_ Male,income_ >50K
0,39,7,77516,9,13,4,1,1,4,2174,0,40,39,1,0
1,50,6,83311,9,13,2,4,0,4,0,0,13,39,1,0
2,38,4,215646,11,9,0,6,1,4,0,0,40,39,1,0
3,53,4,234721,1,7,2,6,0,2,0,0,40,39,1,0
4,28,4,338409,9,13,2,10,5,2,0,0,40,5,0,0


In [ ]:
#feature Engineerring
# Example new features
# Check if 'hours-per-week' exists in the columns:
if 'hours-per-week' in df_encoded.columns:
    df_encoded['work_hours_per_week_ratio'] = df_encoded['hours-per-week'] / df_encoded['age']
else:
    print("Column 'hours-per-week' not found in DataFrame.")
    # If the column has a different name, replace it in the line above.
    # For example: df_encoded['work_hours_per_week_ratio'] = df_encoded['hours_per_week'] / df_encoded['age']

# Check if 'capital-gain' and 'capital-loss' are in the columns before calculating 'capital_diff'
if 'capital-gain' in df_encoded.columns and 'capital-loss' in df_encoded.columns:
    df_encoded['capital_diff'] = df_encoded['capital-gain'] - df_encoded['capital-loss']
else:
    print("Either 'capital-gain' or 'capital-loss' column not found in DataFrame.")
    # If these columns have been one-hot encoded, you'll need to adjust the calculation accordingly.

# Check if the new features were created before trying to access them
if 'work_hours_per_week_ratio' in df_encoded.columns and 'capital_diff' in df_encoded.columns:
    display(df_encoded[['work_hours_per_week_ratio', 'capital_diff']].head())
else:
    print("One or both of the new features were not created successfully.")
    # Investigate why the features were not created (e.g., missing original columns)

Column 'hours-per-week' not found in DataFrame.
Either 'capital-gain' or 'capital-loss' column not found in DataFrame.
One or both of the new features were not created successfully.


In [ ]:
#Transformation for skewed data
import numpy as np
# Check if 'capital-gain' is still in the DataFrame after encoding
if 'capital-gain' in df_encoded.columns:
    # Apply log transformation if the column exists
    df_encoded['log_capital_gain'] = np.log1p(df_encoded['capital-gain'])

    # Check transformation effect
    print(df_encoded[['capital-gain', 'log_capital_gain']].describe())
else:
    print("'capital-gain' column not found. It may have been one-hot encoded.")
    # If it was one-hot encoded, find the new columns and apply the log transformation to them
    capital_gain_cols = [col for col in df_encoded.columns if 'capital-gain' in col]
    if capital_gain_cols:
        for col in capital_gain_cols:
            df_encoded['log_' + col] = np.log1p(df_encoded[col])
        print("Log transformation applied to one-hot encoded 'capital-gain' columns.")
    else:
        print("No 'capital-gain' related columns found in the DataFrame.")

'capital-gain' column not found. It may have been one-hot encoded.
No 'capital-gain' related columns found in the DataFrame.


In [ ]:
#Feature of selection
#Isolation Forest for Outlier Detection
from sklearn.ensemble import IsolationForest

# Apply Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(df_encoded[num_cols])

# Remove outliers
df_filtered = df_encoded[outliers == 1]

# Check dataset shape before and after
print(f"Before: {df_encoded.shape}, After: {df_filtered.shape}")


Before: (32561, 15), After: (30933, 15)


In [ ]:
#PPS (Predictive Power Score) Analysis
import ppscore as pps

# Compute PPS matrix
pps_matrix = pps.matrix(df_filtered)

# Show top feature interactions
pps_matrix.sort_values(by="ppscore", ascending=False).head(10)


,x,y,ppscore,case,is_valid_score,metric,baseline_score,model_score,model
0,age,age,1.0,predict_itself,True,None,0.0,1.0,None
16,workclass,workclass,1.0,predict_itself,True,None,0.0,1.0,None
208,sex_ Male,sex_ Male,1.0,predict_itself,True,None,0.0,1.0,None
192,native_country,native_country,1.0,predict_itself,True,None,0.0,1.0,None
176,hours_per_week,hours_per_week,1.0,predict_itself,True,None,0.0,1.0,None
160,capital_loss,capital_loss,1.0,predict_itself,True,None,0.0,1.0,None
144,capital_gain,capital_gain,1.0,predict_itself,True,None,0.0,1.0,None
128,race,race,1.0,predict_itself,True,None,0.0,1.0,None
96,occupation,occupation,1.0,predict_itself,True,None,0.0,1.0,None
80,marital_status,marital_status,1.0,predict_itself,True,None,0.0,1.0,None


In [ ]:
#cleaned dataset
df_filtered.to_csv("/content/adult_cleaned.csv", index=False)
print("Preprocessed dataset saved!")


Preprocessed dataset saved!
